# BTTH3 — Trên lớp: Voting DApp

**Bài thực hành số 3:** Xây dựng và triển khai ứng dụng phi tập trung (DApp)

**Phạm vi:** phần **trên lớp** (Voting DApp) — **không gồm BTVN** (Digital Health Records).

**Mã nguồn:** thư mục `voting_dapp/` (cùng cấp với notebook).

| Thành phần | Đường dẫn |
|---|---|
| Smart contract | `voting_dapp/contracts/VotingApp.vy` |
| Tests | `voting_dapp/tests/` |
| Scripts | `voting_dapp/scripts/` |
| GUI | `voting_dapp/voting_gui_app.py` |

> Windows: `.venv-ape` (Python 3.12); Geth `--dev --http ... --ipcdisable`.


### i. Thiết lập môi trường

**Bước 1–4:** Nếu lab 1–2 đã cài (`ape`, `vyper`, Geth) thì bỏ qua.

**Bước 5:** Project Ape `voting_dapp` (đã tạo sẵn):

```text
voting_dapp/
  contracts/VotingApp.vy
  scripts/deploy.py
  scripts/add_proposals.py
  scripts/give_right_to_vote.py
  scripts/vote.py
  tests/conftest.py
  tests/test_voting_app.py
  voting_gui_app.py
  ape-config.yaml
```

**Bước 6–7:** Cài PySide6 (nếu chưa) và kiểm tra version (cell code bên dưới).

**Hình 2 — Cấu trúc project:**

![Cây thư mục voting_dapp](Images/hinh_cay_thu_muc_voting_dapp.png)


In [ ]:
from pathlib import Path
import shutil
import subprocess

ROOT = Path(r"d:\Study\IS355")
APE = ROOT / ".venv-ape" / "Scripts" / "ape.exe"
PY = ROOT / ".venv-ape" / "Scripts" / "python.exe"
VYPER = ROOT / ".venv-ape" / "Scripts" / "vyper.exe"

def run(cmd):
    print(">", " ".join(str(c) for c in cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    print(out.strip() or f"(exit {r.returncode})")
    print()

if APE.exists():
    run([str(APE), "--version"])
else:
    print("Không thấy ape.exe — kiểm tra .venv-ape")

if VYPER.exists():
    run([str(VYPER), "--version"])
elif shutil.which("vyper"):
    run(["vyper", "--version"])

if PY.exists():
    run([str(PY), "-c", "import PySide6; print(PySide6.__version__)"])
else:
    print("Không thấy python trong .venv-ape")


### ii. Kiến trúc hệ thống DApp (3 tầng)

1. **Smart Contract (Blockchain):** Vyper + Ape — toàn vẹn dữ liệu bỏ phiếu
2. **Backend Scripts:** Web3/Ape gọi hàm contract, gửi tx, đọc kết quả
3. **Frontend GUI:** PySide6 — user bỏ phiếu / xem kết quả


## 3. Xây dựng Smart Contract (B)

### Bài toán Voting DApp

Hệ thống bỏ phiếu phi tập trung: mọi thao tác (cấp quyền, vote, ủy quyền, công bố kết quả) ghi trên blockchain → khó gian lận.

| Thành phần | Mô tả |
|---|---|
| Voter | Địa chỉ được cấp quyền vote / delegate |
| Proposal | Lựa chọn (ứng viên / phương án) |
| Delegation | Chuyển quyền phiếu cho voter khác |
| Vote | Mỗi người chỉ vote 1 lần |
| Kết quả | `winningProposal()` / `winnerName()` |

### Các hàm chính

| Hàm | Chức năng |
|---|---|
| `giveRightToVote(voter, weight)` | Chairperson cấp quyền + weight |
| `delegate(to)` | Ủy quyền phiếu |
| `vote(proposal)` | Bỏ phiếu theo index |
| `winningProposal()` | Index đề xuất thắng |
| `winnerName()` | Tên đề xuất thắng |
| `addProposal(name)` | Chairperson thêm đề xuất |

### Luồng hoạt động

1. Chairperson deploy + `addProposal`
2. Chairperson `giveRightToVote` cho các địa chỉ
3. User `vote` hoặc `delegate` (qua script / GUI)
4. Đọc `winningProposal` / `winnerName`


### i–iv. Contract `VotingApp.vy`

File đầy đủ nằm ở `voting_dapp/contracts/VotingApp.vy`.

**Lưu ý:** hàm `delegate` gọi `_forwardWeight` (chuyển weight theo chuỗi ủy quyền) — cần để ủy quyền hoạt động đúng; PDF lab có gọi hàm này trong `delegate`.

Cell dưới đây in ra contract để đối chiếu báo cáo.


In [ ]:
from pathlib import Path
p = Path(r"d:\Study\IS355\TH\w3\voting_dapp\contracts\VotingApp.vy")
print(p.read_text(encoding="utf-8"))


### vi. Biên dịch và kiểm thử

Trong thư mục `voting_dapp`:

```text
ape compile
ape test
```

**Nộp báo cáo:** chụp kết quả `ape compile` + địa chỉ contract khi deploy; chụp kết quả `ape test` và giải thích từng test trong `test_voting_app.py`.


In [ ]:
from pathlib import Path
import subprocess

APE = Path(r"d:\Study\IS355\.venv-ape\Scripts\ape.exe")
PROJ = Path(r"d:\Study\IS355\TH\w3\voting_dapp")

for args in (["compile"], ["test", "-q"]):
    cmd = [str(APE), *args]
    print(">", " ".join(cmd))
    r = subprocess.run(cmd, cwd=str(PROJ), capture_output=True, text=True)
    print((r.stdout or "") + (r.stderr or ""))
    print("exit:", r.returncode)
    print("-" * 40)


### Giải thích các test (`test_voting_app.py`)

| Test | Ý nghĩa |
|---|---|
| `test_chairperson` | Deployer = chairperson |
| `test_addProposal` | Chairperson thêm đề xuất OK |
| `test_addProposal_fail` | Account khác **không** thêm được |
| `test_giveRightToVote` | Cấp weight 1 và 9, tăng `voterCount` |
| `test_giveRightToVote_fail` | Non-chairperson gọi sẽ fail |
| `test_vote_fail` | Vote 2 lần → fail |
| `test_vote` | 2 user vote 2 đề xuất, cập nhật state |
| `test_winnerName` | 1–2 vs mountain → winner = `"mountain"` |

> Dán code test / conftest vào báo cáo (hoặc chụp màn hình file).


In [ ]:
from pathlib import Path
base = Path(r"d:\Study\IS355\TH\w3\voting_dapp\tests")
for name in ("conftest.py", "test_voting_app.py"):
    print("=" * 60, name)
    print((base / name).read_text(encoding="utf-8"))


## 4. Triển khai và tương tác (C)

### Start Geth (Windows)

```text
D:\Tools\Geth\geth.exe --dev --http --http.api eth,web3,net --http.addr 127.0.0.1 --http.port 8545 --ipcdisable
```

![Start Geth](Images/hinh_geth_dev_start.png)

Attach:

```text
D:\Tools\Geth\geth.exe attach http://127.0.0.1:8545
```

![Geth attach](Images/hinh_geth_attach.png)

![Balance accounts[0]](Images/hinh_geth_balance_accounts0.png)

Nạp ETH (Bước B):

```js
web3.fromWei(eth.getBalance(eth.accounts[0]), "ether")
eth.sendTransaction({from: eth.accounts[0], to: "0x569F2B82B6e0AF062019ba5BAC3FA6a98e2FA7c4", value: web3.toWei(10, "ether")})
eth.sendTransaction({from: eth.accounts[0], to: "0xf7F4E94C28beC0BCc0082f277b2AAdb7BD149202", value: web3.toWei(10, "ether")})
eth.sendTransaction({from: eth.accounts[0], to: "0x1Cc299d4c374E56E26E1626147fA44113775F99D", value: web3.toWei(10, "ether")})
eth.sendTransaction({from: eth.accounts[0], to: "0x5A94046CD5Eb0736e4AC969581B00cb3100ca326", value: web3.toWei(10, "ether")})
```

![Send 10 ETH](Images/hinh_geth_send_10eth.png)

### Tạo account Ape

```text
cd d:\Study\IS355\TH\w3\voting_dapp
ape accounts generate voter1
ape accounts generate voter2
ape accounts generate voter3
ape accounts generate chairperson
ape accounts list
```

| Alias | Address |
|---|---|
| chairperson | `0x569F2B82B6e0AF062019ba5BAC3FA6a98e2FA7c4` |
| voter1 | `0xf7F4E94C28beC0BCc0082f277b2AAdb7BD149202` |
| voter2 | `0x1Cc299d4c374E56E26E1626147fA44113775F99D` |
| voter3 | `0x5A94046CD5Eb0736e4AC969581B00cb3100ca326` |

![Ape accounts generate](Images/hinh_ape_accounts_generate.png)

Password các account: `123456` (đã điền trong scripts).

### Deploy + proposals + giveRight + vote

```text
cd d:\Study\IS355\TH\w3\voting_dapp
$ape = "d:\Study\IS355\.venv-ape\Scripts\ape.exe"
$net = "ethereum:local:http://127.0.0.1:8545"
& $ape run deploy --network $net
& $ape run add_proposals --network $net
& $ape run give_right_to_vote --network $net
& $ape run vote --network $net
```

Contract đã dùng: `0x943Da320295c096a3608CF7e8Aa962eA5004478d`

![add_proposals](Images/hinh_ape_add_proposals.png)

![give_right](Images/hinh_ape_give_right_to_vote.png)

![vote](Images/hinh_ape_vote.png)


In [ ]:
# Xem nội dung các script deploy / vote
from pathlib import Path
base = Path(r"d:\Study\IS355\TH\w3\voting_dapp\scripts")
for name in ("deploy.py", "add_proposals.py", "give_right_to_vote.py", "vote.py"):
    print("=" * 60, name)
    print((base / name).read_text(encoding="utf-8"))


## 5. Front-end GUI (D)

File: `voting_dapp/voting_gui_app.py` (account `voter3`, password `123456`).

```text
cd d:\Study\IS355\TH\w3\voting_dapp
d:\Study\IS355\.venv-ape\Scripts\python.exe voting_gui_app.py
```

![GUI dropdown](Images/hinh_gui_dropdown.png)

![GUI chọn beach](Images/hinh_gui_vote_beach.png)

**Vote thành công (`Voted OK`, `winnerName = beach`):**

![Voted OK](Images/hinh_gui_voted_ok.png)


In [ ]:
from pathlib import Path
p = Path(r"d:\Study\IS355\TH\w3\voting_dapp\voting_gui_app.py")
print(p.read_text(encoding="utf-8"))


## 6. Checklist nộp bài trên lớp

- [x] Cây thư mục `voting_dapp`
- [x] `ape compile` / `ape test` (8 passed)
- [x] Tạo account + nạp ETH trên Geth
- [x] Deploy VotingApp
- [x] add_proposals / give_right_to_vote / vote
- [x] GUI PySide6 — Voted OK, winnerName = beach

**Không làm ở đây:** BTVN Digital Health Records (`DHR.vy`).


## 7. Ảnh báo cáo

Tất cả ảnh cần nộp đã gắn ở **mục 4** và **mục 5** (folder `Images/`). Không cần chèn thêm.
